This code detects the location of starting point of an office;

Input: List of offices + Corresponding offices 

=> Output: List of offices with coordinate information

In [1]:
import pandas as pd
import os
import cv2
import numpy as np
import json

In [2]:
def Detect_Office(Json,Office):

    NewList=Json['fields']
    Dict=list()
    for d in NewList:
        try:
            newDict={}
            newDict['inferText']=d['inferText']
            newDict['boundingPoly']=d['boundingPoly']
            Dict.append(newDict)
        except KeyError:
            continue

    res = [d
       for d in Dict 
       if (Office[0] == d['inferText'][0]) or (Office[0] == d['inferText']) or (Office == d['inferText']) or (Office[-2:][0] == d['inferText'])]

    if len(res)!=0:
        res = res[0]['boundingPoly']['vertices']
        Edge=max(int(d['x']) for d in res)
        return(Edge)
    else:
        return(None)

def Detect_Office2(Json,Office):
    NewList=Json['fields']
    Dict=list()
    for d in NewList:
        try:
            newDict={}
            newDict['inferText']=d['inferText']
            newDict['boundingPoly']=d['boundingPoly']
            Dict.append(newDict)
        except KeyError:
            continue

    res = [d
       for d in Dict 
       if ( d['inferText'][0] == '〇') or( d['inferText'][0] == '○') or ( d['inferText'][0] == '0')or ( d['inferText'][0] == 'O') or (Office[0] == d['inferText']) or (Office == d['inferText']) or (Office[-2:][0] == d['inferText'])]

    if len(res)!=0:
        res = res[0]['boundingPoly']['vertices']
        Edge=max(int(d['x']) for d in res)
        return(Edge)
    else:
        return(None)

class NpEncoder(json.JSONEncoder):
    def default(self, obj):
        if isinstance(obj, np.integer):
            return int(obj)
        if isinstance(obj, np.floating):
            return float(obj)
        if isinstance(obj, np.ndarray):
            return obj.tolist()
        return super(NpEncoder, self).default(obj)

### CLOVA FUNCTION ###
import requests
import uuid
import time
import json
import cv2
import base64

api_url = 'https://deelieyxuc.apigw.ntruss.com/custom/v1/1972/ebd01bcbf693d069817622e9839e20914143c7d0d8953eddee40e8b0af96c95b/general'
secret_key = 'S1NmVXpYZlJ0cGJ0ZEFRZXVlbkRkaHFReE9FcHNTQ0U='

def Clova(Year,Page):
    path="C:\\Users\\Keitaro Ninomiya\\Box\\Research Notes (keitaro2@illinois.edu)\\Tokyo_Jobs\\Raw_Data\\Splited\\"+Year+"\\"
    with open(path+"Page"+"{:03d}".format(Page)+"\\"+"Page"+"{:03d}".format(Page)+".jpg",'rb') as f:
         file_data = f.read()

    request_json = {
            'images': [
                {
                    'format': 'jpg',
                    'name': 'demo',
                    'data':base64.b64encode(file_data).decode()}],
            'requestId': str(uuid.uuid4()),
            'version': 'V2',
            'timestamp': int(round(time.time() * 1000)),
            'lang':'ja'
            }
    payload = json.dumps(request_json).encode("UTF-8")
    headers = {'X-OCR-SECRET': secret_key,
              'Content-Type': 'application/json'}
    response = requests.request("POST", api_url, headers=headers, data = payload)
    Json=json.loads(response.text)['images'][0]
    
    return Json

def Clova2(Year,Page):
    path="C:\\Users\\Keitaro Ninomiya\\Box\\Research Notes (keitaro2@illinois.edu)\\Tokyo_Jobs\\Raw_Data\\Splited\\"+Year+"\\"
    stream = open(path+"Page"+"{:03d}".format(Page)+"\\"+"Page"+"{:03d}".format(Page)+".jpg", "rb")
    bytes = bytearray(stream.read())
    numpyarray = np.asarray(bytes, dtype=np.uint8)
    img = cv2.imdecode(numpyarray, cv2.IMREAD_UNCHANGED)
    HH,WW=img.shape[:2]
    cropped=img[0:200,0:WW]
    temp_path="C:\\Users\\Keitaro Ninomiya\\Box\\Research Notes (keitaro2@illinois.edu)\\Tokyo_Jobs\\Raw_Data\\Temp\\"
    cv2.imwrite(temp_path+"Temp.jpg",cropped)
    
    with open(temp_path+"Temp.jpg",'rb') as f:
        file_data = f.read()

    request_json = {
            'images': [
                {
                    'format': 'jpg',
                    'name': 'demo',
                    'data':base64.b64encode(file_data).decode()}],
            'requestId': str(uuid.uuid4()),
            'version': 'V2',
            'timestamp': int(round(time.time() * 1000)),
            'lang':'ja'
            }
    payload = json.dumps(request_json).encode("UTF-8")
    headers = {'X-OCR-SECRET': secret_key,
              'Content-Type': 'application/json'}
    response = requests.request("POST", api_url, headers=headers, data = payload)
    Json=json.loads(response.text)['images'][0]
    
    return Json

In [7]:
def hconcat_resize_min(im_list, interpolation=cv2.INTER_CUBIC):
    h_min = min(im.shape[0] for im in im_list)
    im_list_resize = [cv2.resize(im, (int(im.shape[1] * h_min / im.shape[0]), h_min), interpolation=interpolation)
                      for im in im_list]
    return cv2.hconcat(im_list_resize)

def Check(df):
    for n in range(0,len(df)):
        #Extract key info of office
        Row  = df.iloc[n]

        ###Collect Location information###
        Dept=Row["Dept"]
        Office=Row["Office"]
        print('['+str(n)+',"'+Dept+'","'+Office+'"]')
        try:        
            StrPage=dta[Year][Dept][Office]['Starting_Page']
            EndPage=dta[Year][Dept][Office]['Ending_Page']

            path='C:\\Users\\Keitaro Ninomiya\\Box\\Research Notes (keitaro2@illinois.edu)\\Tokyo_Jobs\\Raw_Data\\Splited\\'+Year+'\\'
            stream = open(path+"Page"+"{:03d}".format(StrPage)+"\\"+"Page"+"{:03d}".format(StrPage)+".jpg",'rb')
            bytes = bytearray(stream.read())
            numpyarray = np.asarray(bytes, dtype=np.uint8)
            img = cv2.imdecode(numpyarray, cv2.IMREAD_UNCHANGED)
            HH,WW=img.shape[:2]
            oldimg=img.copy()[0:200,0:WW]

            for Page in range(StrPage+1,EndPage+1):
                path='C:\\Users\\Keitaro Ninomiya\\Box\\Research Notes (keitaro2@illinois.edu)\\Tokyo_Jobs\\Raw_Data\\Splited\\'+Year+'\\'
                stream = open(path+"Page"+"{:03d}".format(StrPage)+"\\"+"Page"+"{:03d}".format(StrPage)+".jpg",'rb') 
                bytes = bytearray(stream.read())
                numpyarray = np.asarray(bytes, dtype=np.uint8)
                newimg = cv2.imdecode(numpyarray, cv2.IMREAD_UNCHANGED)[0:200,0:WW]
                oldimg=np.concatenate((newimg,oldimg), axis=1)
            oldimg=cv2.copyMakeBorder(oldimg,20,20,20,20,cv2.BORDER_CONSTANT)

            StrLoc=int(dta[Year][Dept][Office]['Office_X1'])
            EndLoc=int(dta[Year][Dept][Office]['Office_X2'])
            StrPage=int(dta[Year][Dept][Office]['Starting_Page'])
            EndPage=int(dta[Year][Dept][Office]['Ending_Page'])

            #Start#
            path='C:\\Users\\Keitaro Ninomiya\\Box\\Research Notes (keitaro2@illinois.edu)\\Tokyo_Jobs\\Raw_Data\\Splited\\'+Year+'\\'
            stream = open(path+"Page"+"{:03d}".format(StrPage)+"\\"+"Page"+"{:03d}".format(StrPage)+".jpg",'rb')            
            bytes = bytearray(stream.read())
            numpyarray = np.asarray(bytes, dtype=np.uint8)
            img = cv2.imdecode(numpyarray, cv2.IMREAD_UNCHANGED)

            Annotate=img.copy()[0:200,0:WW]
            cv2.line(Annotate,(StrLoc,0),(StrLoc,HH),(255,0,0),2)
            Annotate=cv2.copyMakeBorder(Annotate,20,20,20,20,cv2.BORDER_CONSTANT)
            oldimg=hconcat_resize_min((Annotate,oldimg))

            #End#
            path='C:\\Users\\Keitaro Ninomiya\\Box\\Research Notes (keitaro2@illinois.edu)\\Tokyo_Jobs\\Raw_Data\\Splited\\'+Year+'\\'
            stream = open(path+"Page"+"{:03d}".format(EndPage)+"\\"+"Page"+"{:03d}".format(EndPage)+".jpg",'rb')            
            bytes = bytearray(stream.read())
            numpyarray = np.asarray(bytes, dtype=np.uint8)
            img = cv2.imdecode(numpyarray, cv2.IMREAD_UNCHANGED)

            Annotate=img.copy()[0:200,0:WW]
            cv2.line(Annotate,(EndLoc,0),(EndLoc,HH),(255,0,0),2)
            Annotate=cv2.copyMakeBorder(Annotate,20,20,20,20,cv2.BORDER_CONSTANT)
            oldimg=hconcat_resize_min((Annotate,oldimg))

            cv2.imshow(Dept+Office,oldimg)
            cv2.waitKey(0)
        except:
            continue
    cv2.destroyAllWindows()
    cv2.waitKey(0)

In [8]:
def Show(n,StrLoc):
    Row=df.iloc[n]
    Page=Row['Page']
    
    path='C:\\Users\\Keitaro Ninomiya\\Box\\Research Notes (keitaro2@illinois.edu)\\Tokyo_Jobs\\Raw_Data\\Splited\\'+Year+'\\'
    stream = open(path+"Page"+"{:03d}".format(EndPage)+"\\"+"Page"+"{:03d}".format(EndPage)+".jpg",'rb')
    bytes = bytearray(stream.read())
    numpyarray = np.asarray(bytes, dtype=np.uint8)
    img = cv2.imdecode(numpyarray, cv2.IMREAD_UNCHANGED)
    
    HH,WW=img.shape[:2]
    cv2.line(img,(StrLoc,0),(StrLoc,HH),(225,0,0),2)
    cv2.imshow('Projection',img)
    cv2.waitKey(0)

#Step 1

Fill in years to refer.
Remember to keep it as a string. NOT as float.

In [9]:
Year=''
Showa=''
path="C:\\Users\\Keitaro Ninomiya\\Box\\Research Notes (keitaro2@illinois.edu)\\Tokyo_Jobs\\Raw_Data\\Splited\\"+Year+"\\"
os.chdir(path)
df = pd.read_csv(r'C:/Users/Keitaro Ninomiya/Box/Research Notes (keitaro2@illinois.edu)/Tokyo_Jobs/Processed_Data/Index/S'+Showa+'.csv')
df=df.drop(df.columns[0], axis=1)

file_path='C:\\Users\\Keitaro Ninomiya\\Box\\Research Notes (keitaro2@illinois.edu)\\Tokyo_Jobs\\Processed_Data\\'+Year+'\\DataFrame.json'
with open(file_path, encoding="utf-8") as f:
    dta = json.loads(f.read())

#Step 2

The following code will extract the location of offices.

In [36]:
#Test code| Version 2#
#Show Working office#
FailedList=[]
for n in range(0,len(df)):
    #Extract key info of office
    Row  = df.iloc[n]
    ExRow= df.iloc[n-1]

    Page=int(Row["Page"])
    Office=Row["Office"]
    Dept=Row['Dept']

    ExPage=int(ExRow["Page"])
    ExOffice=ExRow["Office"]
    ExDept=ExRow['Dept']

    ###Insert Starting page information to motherframe###
    dta[Year][Dept][Office]={}
    dta[Year][Dept][Office]["Starting_Page"]=Page
    print(dta[Year][Dept][Office])

    ###Collect Location information###
    ##Read image for first page##
    img=cv2.imread("Page"+"{:03d}".format(Page)+"\\"+"Page"+"{:03d}".format(Page)+".jpg")
    #Convert to json via CLOVA
    try:
        Json=Clova(Year,Page)
    except:
        print(Office+ 'failed')
        FailedList.append(list((n,Dept,Office)))
        continue

    #Find X coordinate of 'Office'.
    XCoord_Unit=Detect_Office(Json,Office)
    if XCoord_Unit==None:
        #Add to motherframe
        dta[str(Year)][Dept][Office]["Office_X1"]='NA'
        dta[str(Year)][ExDept][ExOffice]["Ending_Page"]='NA'
        dta[str(Year)][ExDept][ExOffice]["Office_X2"]='NA'
        dta[str(Year)][ExDept][ExOffice]["Page_Range"]='NA'
        print(Office+ 'failed')
        FailedList.append(list((n,Dept,Office)))
        continue
    else:
        dta[str(Year)][Dept][Office]["Office_X1"]=XCoord_Unit
        dta[str(Year)][ExDept][ExOffice]["Ending_Page"]=Page
        dta[str(Year)][ExDept][ExOffice]["Office_X2"]=XCoord_Unit
        dta[str(Year)][ExDept][ExOffice]["Page_Range"]=list(range(ExPage,Page+1))       
        HH=img.shape[:2][0]
        print(Office+ 'success: at row'+str(n))

{'Starting_Page': 2}
人事課（S12.12.18）failed
{'Starting_Page': 4}
文書課success
{'Starting_Page': 6}
庶務課success
{'Starting_Page': 7}
企画課success
{'Starting_Page': 8}
財務課success
{'Starting_Page': 9}
統計課success
{'Starting_Page': 20}
都市計画課failed
{'Starting_Page': 24}
会計課success
{'Starting_Page': 26}
公債課success
{'Starting_Page': 27}
主税課success
{'Starting_Page': 34}
徴収課failed
{'Starting_Page': 39}
用度課success
{'Starting_Page': 43}
地理課success
{'Starting_Page': 48}
庶務課failed
{'Starting_Page': 49}
商工課success
{'Starting_Page': 52}
貿易課success
{'Starting_Page': 53}
出張所failed
{'Starting_Page': 54}
農漁課success
{'Starting_Page': 56}
権度課success
{'Starting_Page': 57}
庶務課success
{'Starting_Page': 58}
学務課failed
{'Starting_Page': 60}
社会教育課failed
{'Starting_Page': 66}
体育課success
{'Starting_Page': 69}
視学課failed
{'Starting_Page': 75}
庶務課failed
{'Starting_Page': 76}
保護課success
{'Starting_Page': 90}
福利課success
{'Starting_Page': 94}
職業課success
{'Starting_Page': 101}
庶務課success
{'Starting_Page': 102}
衛生課success
{'Starti

FileNotFoundError: [Errno 2] No such file or directory: 'C:\\Users\\Keitaro Ninomiya\\Box\\Research Notes (keitaro2@illinois.edu)\\Tokyo_Jobs\\Raw_Data\\Splited\\1938\\Page126\\Page126.jpg'

#Step 3

Check for errors.
Manually type in the index, department name, and office name of erroneous department-office pair to a seperate list.

In [ ]:
Check(df)

Type in the department-offices with errors.

Type1=[]

#Step 4

See how much errors were observed in the first trial.

In [37]:
FailRate=len(FailedList)/len(df)
if len(FailedList)/len(df)<0.2:
    print('Fantastic!! Success Rate is '+str(1-(len(FailedList)/len(df))))
else:
    print('残念...Success Rate is '+str(1-(len(FailedList)/len(df))))
DF=pd.read_csv('C:\\Users\\Keitaro Ninomiya\\Box\\Research Notes (keitaro2@illinois.edu)\\Tokyo_Jobs\\Processed_Data\\Records.csv')
DF.loc[int(Year)-1934, 'Office'] = 1-FailRate
display(DF)
DF.to_csv('C:\\Users\\Keitaro Ninomiya\\Box\\Research Notes (keitaro2@illinois.edu)\\Tokyo_Jobs\\Processed_Data\\Records.csv')

{'1938': {'Admin': {'秘書課': {'Starting_Page': 2,
    'Office_X1': 479,
    'Ending_Page': 'NA',
    'Office_X2': 'NA',
    'Page_Range': 'NA'},
   '人事課（S12.12.18）': {'Starting_Page': 2,
    'Office_X1': 'NA',
    'Ending_Page': 4,
    'Office_X2': 377,
    'Page_Range': [2, 3, 4]},
   '文書課': {'Starting_Page': 4,
    'Office_X1': 367,
    'Ending_Page': 6,
    'Office_X2': 145,
    'Page_Range': [4, 5, 6]}},
  '中央卸売市場': {},
  '企画局（S12.12.18）': {'庶務課': {'Starting_Page': 6,
    'Office_X1': 135,
    'Ending_Page': 7,
    'Office_X2': 255,
    'Page_Range': [6, 7]},
   '企画課': {'Starting_Page': 7,
    'Office_X1': 245,
    'Ending_Page': 8,
    'Office_X2': 217,
    'Page_Range': [7, 8]},
   '財務課': {'Starting_Page': 8,
    'Office_X1': 207,
    'Ending_Page': 9,
    'Office_X2': 129,
    'Page_Range': [8, 9]},
   '統計課': {'Starting_Page': 9,
    'Office_X1': 119,
    'Ending_Page': 'NA',
    'Office_X2': 'NA',
    'Page_Range': 'NA'},
   '都市計画課': {'Starting_Page': 20,
    'Office_X1': 'NA',
 

In [39]:
#Fixing Failed Offices
#Step1: Check for simple page assignment error
path="C:\\Users\\Keitaro Ninomiya\\Box\\Research Notes (keitaro2@illinois.edu)\\Tokyo_Jobs\\Raw_Data\\Splited\\"+Year+"\\"
for n in range(0,len(FailedList)):
    Dept=FailedList[n][0]
    Office=FailedList[n][1]
    print(Dept,Office)
    Page=df['Page'][(df['Office']==Office) & (df['Dept']==Dept)].tolist()[0]
    image=cv2.imread(path+"Page"+"{:03d}".format(Page)+"\\"+"Page"+"{:03d}".format(Page)+".jpg")
    cv2.imshow('Image',image)
    cv2.waitKey(0)

経理局（S12.12.18） 徴収課
産業局 庶務課
産業局 出張所
教育局 学務課
教育局 社会教育課
教育局 視学課
社会局 庶務課
保健局 公園課


Fix the department-office pair with errors

First re-run the list of failed (pairs which was rejected), with a stricter OCR.

In [ ]:
FailedList2=[]
for Office in FailedList:
    n=Office[0]
    Row=df.iloc[n]
    ExRow= df.iloc[n-1]

    Page=int(Row["Page"])
    Office=Row["Office"]
    Dept=Row['Dept']

    ExPage=int(ExRow["Page"])
    ExOffice=ExRow["Office"]
    ExDept=ExRow['Dept']

    Json=Clova2(Year,Page)
    
    try:
        XCoord_Unit=Detect_Office2(Json,Office)
        dta[str(Year)][Dept][Office]["Office_X1"]=XCoord_Unit
        dta[str(Year)][ExDept][ExOffice]["Ending_Page"]=Page
        dta[str(Year)][ExDept][ExOffice]["Office_X2"]=XCoord_Unit+10
        dta[str(Year)][ExDept][ExOffice]["Page_Range"]=list(range(ExPage,Page+1))       
        print(Office+ 'success: at row '+str(n))
        img=cv2.imread(path+"Page"+"{:03d}".format(Page)+"\\"+"Page"+"{:03d}".format(Page)+".jpg")
        #cv2.line(img, (XCoord_Unit,0), (XCoord_Unit,300), (255,0,0), 2)
        #cv2.imshow('pic',img)
        #cv2.waitKey(0)
    except:
        FailedList2.append(list((n,Dept,Office)))
        continue

Open FailedList_2 and see if there are any other errors left.

If there are, fix it manually by guessing the starting x axis using 'Show(n,StrLoc)'

When you find the exact starting location, update the dataframe.

dta[Year][Dept][Office]={'Starting_Page': ,
                          'Office_X1': ,
                          'Ending_Page': ,
                          'Office_X2': ,
                          'Page_Range': []}

#Step 5

Do the same thing with the department-office pair which was wrongly accepted in the test.

In [ ]:
Type1_2=[]
for Office in Type1:
    n=Office[0]
    Row=df.iloc[n]
    ExRow= df.iloc[n-1]

    Page=int(Row["Page"])
    Office=Row["Office"]
    Dept=Row['Dept']

    ExPage=int(ExRow["Page"])
    ExOffice=ExRow["Office"]
    ExDept=ExRow['Dept']

    Json=Clova2(Year,Page)
    
    try:
        XCoord_Unit=Detect_Office2(Json,Office)
        dta[str(Year)][Dept][Office]["Office_X1"]=XCoord_Unit
        dta[str(Year)][ExDept][ExOffice]["Ending_Page"]=Page
        dta[str(Year)][ExDept][ExOffice]["Office_X2"]=XCoord_Unit+10
        dta[str(Year)][ExDept][ExOffice]["Page_Range"]=list(range(ExPage,Page+1))       
        print(Office+ 'success: at row '+str(n))
        img=cv2.imread(path+"Page"+"{:03d}".format(Page)+"\\"+"Page"+"{:03d}".format(Page)+".jpg")
        #cv2.line(img, (XCoord_Unit,0), (XCoord_Unit,300), (255,0,0), 2)
        #cv2.imshow('pic',img)
        #cv2.waitKey(0)
    except:
        Type1_2.append(list((n,Dept,Office)))
        continue

#Step 6

Check the entire dataframe to confirm that the lines have been drawn at the correct place.

If there are errors, add the pair into the Type1 Error list and re-run step 5.

In [ ]:
Check(df)

In [ ]:
json_object = json.dumps(dta, indent=4,
                        cls=NpEncoder)
save_path='C:\\Users\\Keitaro Ninomiya\\Box\\Research Notes (keitaro2@illinois.edu)\\Tokyo_Jobs\\Processed_Data\\'+str(Year)+'\\'
with open(save_path+"DataFrame.json", "w") as outfile:
    outfile.write(json_object)